**1- Importing the Required Libraries**

We import the libraries needed for data analysis, machine learning, visualization, and statistical plotting: Pandas, NumPy, Scikit-learn, Matplotlib, and Seaborn.

In [1]:
import sys
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
pd.set_option('display.float_format', lambda x: '%.2f' % x)


In [2]:
from sklearn.model_selection import train_test_split 

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [3]:

sys.path.append(os.path.abspath('..'))

from config.path import DATA_PROCESSED_PATH, HYPERPARAMETER_OPTIMIZATION_RESULTS_PATH

In [4]:
df = pd.read_csv(DATA_PROCESSED_PATH)
df.head()


,year,selling_price,km_driven,owner,fuel_CNG,fuel_Diesel,fuel_Electric,fuel_LPG,fuel_Petrol,fuel_nan,transmission_Automatic,transmission_Manual,transmission_nan,seller_type_Dealer,seller_type_Individual,seller_type_Trustmark Dealer,seller_type_nan,brand
0,2007.00,60000,70000.00,1,False,False,False,False,True,False,False,True,False,False,True,False,False,18
1,2007.00,135000,50000.00,0,False,False,False,False,True,False,False,True,False,False,True,False,False,18
2,2012.00,600000,100000.00,1,False,True,False,False,False,False,False,True,False,False,True,False,False,10
3,2017.00,250000,46000.00,1,False,False,False,False,True,False,False,True,False,False,True,False,False,5
4,2014.00,450000,141000.00,2,False,True,False,False,False,False,False,True,False,False,True,False,False,9


In [5]:
X = df.drop(columns=['selling_price'])
Y = df['selling_price']

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)


In [6]:

rf_param_grid = {
    'model__n_estimators': [100, 200, 300], 
    'model__max_depth': [None,5, 10, 15, 20] , 
    'model__min_samples_split': [2,5,10] , 
    'model__min_samples_leaf' : [1,2,4],
}
xgb_param_grid = {
    'model__n_estimators' : [100, 200, 300] , 
    'model__learning_rate' : [0.01, 0.05, 0.1, 0.2] , 
    'model__max_depth' : [3,6,10] , 
    'model__subsample' : [0.6, 0.8, 1.0] ,
    'model__colsample_bytree' : [0.6 , 0.8, 1.0]
}


In [7]:
rf_pipeline = Pipeline([
    ('scaler', StandardScaler() ) , 
    ('model' , RandomForestRegressor(random_state=42))
])

xgb_pipeline = Pipeline([
    ('scaler', StandardScaler() ) , 
    ('model' , XGBRegressor(random_state=42))
])


rf_grid = GridSearchCV(
    estimator=rf_pipeline, 
    param_grid=rf_param_grid, 
    cv=5, 
    scoring='r2' , 
    n_jobs=-1
)

xgb_grid = GridSearchCV(
    estimator=xgb_pipeline, 
    param_grid=xgb_param_grid, 
    cv=5, 
    scoring='r2' , 
    n_jobs=-1
)




In [8]:
print("Training Random Forest model...")
rf_grid.fit(X_train, y_train)

print("Training XGBoost model...")
xgb_grid.fit(X_train, y_train)

print('=' * 50)

print("Best parameters for Random Forest:", rf_grid.best_params_)
print("Best parameters for XGBoost:", xgb_grid.best_params_)

print('=' * 50)

print("Best cross-validation scores for Random Forest:", rf_grid.best_score_)
print("Best cross-validation scores for XGBoost:", xgb_grid.best_score_)

Training Random Forest model...
Training XGBoost model...
Best parameters for Random Forest: {'model__max_depth': 10, 'model__min_samples_leaf': 1, 'model__min_samples_split': 5, 'model__n_estimators': 200}
Best parameters for XGBoost: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.05, 'model__max_depth': 6, 'model__n_estimators': 200, 'model__subsample': 0.6}
Best cross-validation scores for Random Forest: 0.6230269686669943
Best cross-validation scores for XGBoost: 0.6475561261177063


In [9]:
rf_best = rf_grid.best_estimator_
xgb_best = xgb_grid.best_estimator_


rf_pred = rf_best.predict(X_test) 
xgb_pred = xgb_best.predict(X_test)


result = []

result.append({
    'Model': 'Random Forest',
    'MSE': mean_squared_error(y_test, rf_pred) , 
    'MAE': mean_absolute_error(y_test, rf_pred) ,
    'RMSE' : np.sqrt(mean_squared_error(y_test, rf_pred)),
    'R2' : r2_score(y_test, rf_pred)
})

result.append({
    'Model': 'XGBoost',
    'MSE': mean_squared_error(y_test, xgb_pred) ,
    'MAE': mean_absolute_error(y_test, xgb_pred) ,
    'RMSE' : np.sqrt(mean_squared_error(y_test, xgb_pred)) ,
    'R2' : r2_score(y_test, xgb_pred)
})

results_df = pd.DataFrame(result)

results_df.to_csv(HYPERPARAMETER_OPTIMIZATION_RESULTS_PATH, index=False)